# Connection with local DataBase

In [ ]:
from sqlalchemy import create_engine
import urllib
import pandas as pd

# Your existing pyodbc connection details
server = r"MELKON\SQLEXPRESS"
database = "DataWarehouse"

# Construct ODBC connection string
odbc_str = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;TrustServerCertificate=yes;"

# Encode for SQLAlchemy
connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"

# Create SQLAlchemy engine
engine = create_engine(connection_url)

**Script Overview:**

======================================================================================================================
This SQL script updates the `silver.crm_cust_info` table by adding an `is_future` column and inserting the latest 
cleaned customer records from `bronze.crm_cust_info`. It ensures data consistency by:  

- Selecting the most recent record per customer (`cust_id`) using `ROW_NUMBER()`.  
- Cleaning names and standardizing marital status and gender values.  
- Flagging future-dated records in the `is_future` column.  

This script is part of an ETL process, ensuring accurate and up-to-date customer data for analytics and reporting. ??

=======================================================================================================================

In [ ]:
# SQL Query
query = """
SELECT
    cust_id,
    cust_key,
    TRIM(cust_firstname) AS cust_firstname,
    TRIM(cust_lastname) AS cust_lastname,
    CASE 
        WHEN UPPER(TRIM(cust_marital_status)) = 'M' THEN 'Married'
        WHEN UPPER(TRIM(cust_marital_status)) = 'S' THEN 'Single'
        ELSE 'n/a'
    END AS cust_marital_status,
    CASE 
        WHEN UPPER(TRIM(cust_gender)) = 'M' THEN 'Male'
        WHEN UPPER(TRIM(cust_gender)) = 'F' THEN 'Female'
        ELSE 'n/a'
    END AS cust_gender,
    cust_create_date,
    CASE 
        WHEN cust_create_date IS NULL THEN 0
        WHEN cust_create_date > GETDATE() THEN 1
        ELSE 0
    END AS cust_is_future
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY cust_create_date DESC) AS flag_last
    FROM bronze.crm_cust_info
    WHERE cust_id IS NOT NULL
) latest_record
WHERE flag_last = 1;
"""

# Execute query using SQLAlchemy engine
df_results = pd.read_sql(query, engine)

# Display DataFrame
df_results

In [ ]:
df = df_results
print(df.head())

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# Database Connection
server = r"MELKON\SQLEXPRESS"
database = "DataWarehouse"

# Construct ODBC connection string
odbc_str = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;TrustServerCertificate=yes;"
connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"

# Create SQLAlchemy engine
engine = create_engine(connection_url)

# Step 1: Get all tables in 'bronze' schema
query_get_tables = """
SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'bronze'
"""
df_tables = pd.read_sql(query_get_tables, engine)
table_names = df_tables['TABLE_NAME'].tolist()

# Step 2: Iterate through each table
for table_name in table_names:
    print(f"\n?? Processing Table: {table_name}\n")

    # Step 3: Get all column names for the current table
    query_get_columns = f"""
    SELECT COLUMN_NAME, DATA_TYPE 
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'bronze' AND TABLE_NAME = '{table_name}'
    """
    
    df_columns = pd.read_sql(query_get_columns, engine)
    column_names = df_columns['COLUMN_NAME'].tolist()
    column_data_types = dict(zip(df_columns['COLUMN_NAME'], df_columns['DATA_TYPE']))

    # Step 4: Generate dynamic SELECT statement with cleansing
    select_clauses = []
    
    for col in column_names:
        col_lower = col.lower()
        data_type = column_data_types[col]

        # 1?? Trim Strings
        if data_type in ['varchar', 'nvarchar']:
            select_clauses.append(f"TRIM({col}) AS {col}")

        # 2?? Standardize Material Status
        elif "marital_status" in col_lower:
            select_clauses.append(f"""
                CASE 
                    WHEN UPPER(TRIM({col})) = 'M' THEN 'Married'
                    WHEN UPPER(TRIM({col})) = 'S' THEN 'Single'
                    ELSE 'n/a'
                END AS {col}
            """)

        # 3?? Standardize Gender
        elif "gender" in col_lower:
            select_clauses.append(f"""
                CASE 
                    WHEN UPPER(TRIM({col})) = 'M' THEN 'Male'
                    WHEN UPPER(TRIM({col})) = 'F' THEN 'Female'
                    ELSE 'n/a'
                END AS {col}
            """)

        # 4?? Handle Future Dates
        elif "create_date" in col_lower and data_type in ['datetime', 'date']:
            select_clauses.append(f"""
                CASE 
                    WHEN {col} IS NULL THEN 0
                    WHEN {col} > GETDATE() THEN 1
                    ELSE 0
                END AS cust_is_future
            """)

        # 5?? Keep Everything Else as is
        else:
            select_clauses.append(f"{col}")

    # Convert list into SQL SELECT statement
    select_query = ",\n    ".join(select_clauses)

    # Step 5: Execute the dynamically generated query
    query_clean = f"""
    SELECT
        {select_query}
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY cust_create_date DESC) AS flag_last
        FROM bronze.{table_name}
        WHERE cust_id IS NOT NULL
    ) latest_record
    WHERE flag_last = 1;
    """

    # Execute query and load into DataFrame
    df_cleaned = pd.read_sql(query_clean, engine)

    # Print the cleaned table for each iteration
    print(df_cleaned)

print("\n? Cleaning process completed for all tables in 'bronze' schema!")
